# Gộp Dữ Liệu Luồng 1 (Tính Nhất Quán) và Luồng 2 (Chỉ Số ESG Báo Cáo)

Tập lệnh này sẽ tự động tìm kiếm và đối chiếu 2 thư mục Output từ `Thread_1` và `Thread_2`. Nó sẽ gộp các ngân hàng có đầy đủ cả 2 file kết quả:
- **Thread 1:** `Thread_1/data/{year}/{bank}/scoring/{bank}_Gap_Score.xlsx`
- **Thread 2:** `Thread_2/outputs/{year}/{bank}/GSI_Result.json`

Sau đó xuất ra 1 file duy nhất với 7 tính năng (features) làm dữ liệu gốc cho K-Means chạy ở luồng 3.

In [3]:
import pandas as pd
import json
import os
from pathlib import Path

# =====================================================================
# CẤU HÌNH ĐƯỜNG DẪN 
# =====================================================================
PROJECT_ROOT = Path("D:/NCKH")

# Input từ luồng 1 và 2
T1_DIR = PROJECT_ROOT / "Thread_1" / "data"
T2_DIR = PROJECT_ROOT / "Thread_2" / "outputs"

# Thư mục lưu kết quả luồng 3
T3_DIR = PROJECT_ROOT / "Thread_3"
T3_DIR.mkdir(parents=True, exist_ok=True)

# File đích lưu gộp
FINAL_OUTPUT_FILE = T3_DIR / "Combined_Features_For_KMeans.xlsx"

In [4]:
# =====================================================================
# THỰC THI GỘP DỮ LIỆU
# =====================================================================
combined_data = []

# Duyệt qua các năm trong luồng 1 (vd: 2023, 2024...)
if T1_DIR.exists():
    for year_dir in T1_DIR.iterdir():
        if not year_dir.is_dir():
            continue
            
        year = year_dir.name
        
        # Duyệt qua các ngân hàng trong từng năm (vd: ACB, BIDV...)
        for bank_dir in year_dir.iterdir():
            if not bank_dir.is_dir():
                continue
                
            bank_name = bank_dir.name
            
            # Tạo đường dẫn động dựa trên cấu trúc đã thu thập được
            t1_file = bank_dir / "scoring" / f"{bank_name}_Gap_Score.xlsx"
            t2_file = T2_DIR / year / bank_name / "GSI_Results.json"
            
            # Chỉ xử lý khi ngân hàng này CÓ ĐẦY ĐỦ output từ cả 2 luồng
            if t1_file.exists() and t2_file.exists():
                try:
                    # 1. Đọc dữ liệu từ luồng 1 (3 Features)
                    df_t1 = pd.read_excel(t1_file)
                    
                    report_x = float(df_t1['Report_Score_X'].values[0])
                    news_y   = float(df_t1['News_Score_Y_mean'].values[0])
                    delta    = float(df_t1['Delta_Gap'].values[0])
                    
                    # 2. Đọc dữ liệu từ luồng 2 (4 Features)
                    with open(t2_file, 'r', encoding='utf-8') as f:
                        data_t2 = json.load(f)
                        
                    e_score = float(data_t2['ESG Scores']['E_tfidf'])
                    s_score = float(data_t2['ESG Scores']['S_tfidf'])
                    g_score = float(data_t2['ESG Scores']['G_tfidf'])
                    gsi_raw = float(data_t2['GSI Score']['GSI_Raw'])
                    
                    # 3. Gộp thành 1 dòng (Row)
                    combined_data.append({
                        "Year": year,
                        "Bank_Name": bank_name,
                        "Report_Score_X": report_x,
                        "News_Score_Y_mean": news_y,
                        "Delta_Gap": delta,
                        "E_tfidf": e_score,
                        "S_tfidf": s_score,
                        "G_tfidf": g_score,
                        "GSI_Raw": gsi_raw
                    })
                    print(f"✅ Gộp thành công: {bank_name} ({year})")
                    
                except Exception as e:
                    print(f"❌ Lỗi khi đọc dữ liệu của {bank_name} ({year}): {e}")
            else:
                # Cảnh báo rớt dữ liệu
                if not t1_file.exists():
                    print(f"⚠️ Thiếu file Luồng 1 cho {bank_name} ({year}): {t1_file}")
                if not t2_file.exists():
                    print(f"⚠️ Thiếu file Luồng 2 cho {bank_name} ({year}): {t2_file}")

# Xuất ra file Excel
if combined_data:
    df_combined = pd.DataFrame(combined_data)
    df_combined.to_excel(FINAL_OUTPUT_FILE, index=False)
    print("\n" + "="*60)
    print(f"🎉 HOÀN TẤT! Đã gộp thành công {len(combined_data)} ngân hàng.")
    print(f"📁 Dữ liệu được lưu tại: {FINAL_OUTPUT_FILE}")
    print("="*60)
    display(df_combined.head())
else:
    print("❌ Không gộp được bất kỳ ngân hàng nào. Vui lòng kiểm tra lại sự tồn tại của các file output ở 2 luồng.")

✅ Gộp thành công: ACB (2023)
✅ Gộp thành công: Agribank (2023)
✅ Gộp thành công: BIDV (2023)
⚠️ Thiếu file Luồng 2 cho HDBank (2023): D:\NCKH\Thread_2\outputs\2023\HDBank\GSI_Results.json
✅ Gộp thành công: MSB (2023)
✅ Gộp thành công: NamABank (2023)
✅ Gộp thành công: OCB (2023)
✅ Gộp thành công: PVcomBank (2023)
✅ Gộp thành công: SHB (2023)
✅ Gộp thành công: Techcombank (2023)
✅ Gộp thành công: TPBank (2023)
✅ Gộp thành công: Vietcombank (2023)
✅ Gộp thành công: VietinBank (2023)
✅ Gộp thành công: VPBank (2023)
✅ Gộp thành công: ACB (2024)
✅ Gộp thành công: Agribank (2024)
✅ Gộp thành công: BIDV (2024)
⚠️ Thiếu file Luồng 2 cho HDBank (2024): D:\NCKH\Thread_2\outputs\2024\HDBank\GSI_Results.json
✅ Gộp thành công: NCB (2024)
✅ Gộp thành công: OCB (2024)
✅ Gộp thành công: PVcomBank (2024)
✅ Gộp thành công: SHB (2024)
✅ Gộp thành công: Techcombank (2024)
✅ Gộp thành công: TPBank (2024)
✅ Gộp thành công: VietAbank (2024)
✅ Gộp thành công: Vietcombank (2024)
✅ Gộp thành công: VietinBank (2

,Year,Bank_Name,Report_Score_X,News_Score_Y_mean,Delta_Gap,E_tfidf,S_tfidf,G_tfidf,GSI_Raw
0,2023,ACB,0.027297,0.014135,0.013161,0.026424,0.007108,0.017820,0.017117
1,2023,Agribank,0.010542,0.008809,0.001733,0.020279,0.005499,0.018529,0.014769
2,2023,BIDV,0.020037,0.005126,0.014911,0.017630,0.005169,0.013448,0.012082
3,2023,MSB,0.021417,0.000000,0.021417,0.015391,0.004072,0.016851,0.012105
4,2023,NamABank,0.020782,0.004592,0.016190,0.018583,0.004685,0.017899,0.013722
